In [ ]:
# Brute-force Detection Logic

from collections import defaultdict
from datetime import datetime
 
THRESHOLD   = 5    # number of failures to trigger
WINDOW_SECS = 60   # sliding window in seconds
 
# Group 401 timestamps by IP
fails = defaultdict(list)   # { '198.51.100.17': [datetime, datetime, ...] }
 
# For each IP, check every window of THRESHOLD events
for ip, times in fails.items():
    times.sort()
    for i in range(len(times) - THRESHOLD + 1):
        window = (times[i + THRESHOLD - 1] - times[i]).total_seconds()
        if window <= WINDOW_SECS:
            print(f'[ALERT] Brute-force: {ip}')  # fire once per IP
            break


In [ ]:
# Line 16 creates a new user with UID=0 — root-level privilege. This is a high-severity IOC your script must detect. from auth.log

In [ ]:
#Reading EVE JSON in Python
import json
from pathlib import Path
 
events = []
for line in Path('eve.json').read_text().splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    try:
        events.append(json.loads(line))   # each line is one dict
    except json.JSONDecodeError:
        pass   # skip malformed lines
 
# Filter to alert events only
alerts = [e for e in events if e.get('event_type') == 'alert']
 
# Access nested fields with dict notation
for a in alerts:
    print(a['alert']['severity'], a['alert']['signature'])


#Never use json.load() on an EVE file — that function expects the entire file to be one JSON value.
#EVE is NDJSON (Newline-Delimited JSON): one object per line. Always iterate lines and call json.loads() on each.


In [ ]:
# Reading EVE JSON

import json
from pathlib import Path
 
events = []
for line in Path('eve.json').read_text().splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    try:
        events.append(json.loads(line))   # each line is one dict
    except json.JSONDecodeError:
        pass   # skip malformed lines
 
# Filter to alert events only
alerts = [e for e in events if e.get('event_type') == 'alert']
 
# Access nested fields with dict notation
for a in alerts:
    print(a['alert']['severity'], a['alert']['signature'])


In [ ]:
# Your goal is to extract all attribute values into a flat Python set for fast lookup during log scanning. 
# parsing - Navigate the nested structure: response list → Event → Attribute list → value field. Filter by the type field to target specific IOC categories. 
import json 

WANTED_TYPES = {'ip-dst', 'domain', 'md5', 'sha256', 'url', 'email-src'} 

def load_misp(path: str) -> set[str]: 
    iocs: set[str] = set() 
    data = json.loads(open(path).read()) 

    for item in data.get('response', []): 
        for attr in item.get('Event', {}).get('Attribute', []): 
            if attr.get('type') in WANTED_TYPES: 
                iocs.add(attr['value'].lower())   # normalise to lowercase 

    return iocs 

  

# Expected result: 

# {'185.220.101.5', 'malware-c2.evil.com', 'd41d8cd98f...', ...} 

In [ ]:
# parsing
# Iterate bundle['objects'], keep only items where type == 'indicator', then extract the value from inside the pattern string using regex. 

import json, re 

  

# Extracts the value from patterns like: [ipv4-addr:value = '1.2.3.4'] 

PATTERN_RE = re.compile(r"\[[\w:.-]+ = '([^']+)'\]") 

  

def load_stix(path: str) -> set[str]: 

    iocs: set[str] = set() 

    bundle = json.loads(open(path).read()) 

    for obj in bundle.get('objects', []): 

        if obj.get('type') != 'indicator': 

            continue 

        m = PATTERN_RE.search(obj.get('pattern', '')) 

        if m: 

            iocs.add(m.group(1).lower()) 
            return iocs

In [ ]:
Build alerting_engine.py — a script that reads all three log sources, loads two threat feeds, and prints a prioritised alert report. 

Expected run-time: 90 minutes. 

No external libraries required — only re, json, pathlib, collections, datetime. 

lab4/ 

├── alerting_engine.py   ← your script (start from scratch) 

├── apache_access.log    ← from Section 2 

├── auth.log             ← from Section 3 

├── eve.json             ← from Section 4 

├── misp_event.json      ← from Section 6.1 

└── threat_bundle.json   ← from Section 6.2 